[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/01_basics.ipynb)

# 教程 1：MuJoCo 基础

学习基础知识：加载模型、`MjModel`、`MjData` 与命名访问。

**前置要求：** [教程 00：什么是 MuJoCo？](00_what_is_mujoco.ipynb)——你应该已经理解：
- “描述 → 仿真 → 观察”模式
- `MjModel` = 蓝图（恒定不变），`MjData` = 状态（会变化）
- `qpos`/`qvel` = 所有关节（joint）的位置/速度

本教程将更深入地讲解如何**检查**模型、如何按名称访问数据。

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media

print(f"MuJoCo version: {mujoco.__version__}")

## 加载模型

MuJoCo 模型用 [MJCF](https://mujoco.readthedocs.io/en/stable/XMLreference.html)（XML 格式）定义。
最小的合法模型就是 `<mujoco/>`。我们来定义一个简单的场景：

In [ ]:
xml = """
<mujoco>
  <worldbody>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)
print(f"Number of geoms: {model.ngeom}")

## MjModel——静态描述

`MjModel` 包含仿真过程中**不会改变**的一切：
几何尺寸、质量、关节类型、颜色等。

In [ ]:
# Number of geoms and their RGBA colors
print(f"ngeom: {model.ngeom}")
print(f"geom_rgba:\n{model.geom_rgba}")

## 命名访问

Python 绑定提供了便捷的命名访问器：

In [ ]:
# Access by name
print("green_sphere rgba:", model.geom('green_sphere').rgba)
print("green_sphere id:", model.geom('green_sphere').id)
print("geom 0 name:", model.geom(0).name)

# List all geom names
print("All geoms:", [model.geom(i).name for i in range(model.ngeom)])

## MjData——动态状态

`MjData` 包含仿真**状态**（时间、位置、速度）
以及派生量（笛卡尔位置、力等）。

In [ ]:
data = mujoco.MjData(model)

# Derived quantities need explicit computation
print("Before mj_kinematics:", data.geom_xpos)

mujoco.mj_kinematics(model, data)
print("After mj_kinematics:", data.geom_xpos)
print("green_sphere pos:", data.geom('green_sphere').xpos)

## 关键要点

- `MjModel` = 静态描述（由 XML 编译而来）
- `MjData` = 动态状态 + 派生量
- 读取派生量之前，先用 `mj_forward()` 或 `mj_kinematics()` 把状态算出来
- 命名访问：`model.geom('name')`、`data.body('name')` 等